In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("../outputs/charts", exist_ok=True)
sns.set_style("whitegrid")

In [ ]:
raw = pd.read_csv("../data/winequality-white.csv", sep=";")
print("Raw data shape:", raw.shape)

In [ ]:
raw.isnull().sum()

In [ ]:
n_dup = int(raw.duplicated().sum())
print("Duplicate rows (exact repeats of an earlier row):", n_dup, "(%.1f%% of all rows)" % (n_dup / len(raw) * 100))
display(raw[raw.duplicated(keep=False)].sort_values(list(raw.columns)).head(6))

In [ ]:
numeric = raw.select_dtypes(include="number")

checks = {
    "Negative values (any column)": int((numeric < 0).sum().sum()),
    "pH outside the 0-14 scale": int((~raw["pH"].between(0, 14)).sum()),
    "Alcohol outside 0-100 %": int((~raw["alcohol"].between(0, 100)).sum()),
    "Free SO2 greater than total SO2": int((raw["free sulfur dioxide"] > raw["total sulfur dioxide"]).sum()),
    "Quality outside the 0-10 scale": int((~raw["quality"].between(0, 10)).sum()),
    "Citric acid exactly 0 (possible, kept)": int((raw["citric acid"] == 0).sum()),
}
pd.Series(checks, name="Count")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.ravel(), raw.columns):
    sns.boxplot(y=raw[col], ax=ax, color="lightsteelblue")
    ax.set_title(col)
    ax.set_ylabel("")
plt.suptitle("Boxplots of all variables (raw data)", y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/charts/boxplots_all_features.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
Q1, Q3 = raw.quantile(0.25), raw.quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((raw < Q1 - 1.5 * IQR) | (raw > Q3 + 1.5 * IQR)).sum()

pd.DataFrame({
    "IQR outliers": outlier_counts,
    "% of rows": (outlier_counts / len(raw) * 100).round(2),
    "min": raw.min(),
    "max": raw.max(),
})

 Decisions and justification

Duplicates - removed. The 937 duplicate rows are exact repeats of an earlier row. If a repeated row lands in both the training and the testing set, the model is tested on data it has already seen, which inflates the reported accuracy. Removing duplicates gives a more honest evaluation. (The cost: a few genuinely different wines may share identical measurements, but this is rare compared with the leakage risk.)

Outliers - kept. Many variables show points beyond the 1.5 x IQR whiskers (for example chlorides, citric acid, residual sugar and free sulfur dioxide). These are real laboratory measurements within plausible ranges (e.g. very sweet wines have a high residual sugar), not data-entry errors, and removing them would delete valid, informative wines. Tree-based models are not sensitive to outliers. The sensitivity of linear models (OLS, Logistic Regression) is noted as a limitation in notebooks 04 to 06.

Missing values / invalid values: none, so no imputation or correction is needed.

In [ ]:
print(raw.dtypes)
print()
print("Quality counts (raw data):")
print(raw["quality"].value_counts().sort_index())

plt.figure(figsize=(7, 4))
sns.countplot(data=raw, x="quality", color="steelblue")
plt.title("Distribution of Wine Quality (raw data)")
plt.xlabel("Quality score")
plt.ylabel("Number of wines")
plt.savefig("../outputs/charts/target_distribution_raw.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)

quality_report = pd.DataFrame([
    ("Original row count",                      len(raw)),
    ("Duplicate rows originally",               int(raw.duplicated().sum())),
    ("Duplicate rows removed",                  len(raw) - len(df)),
    ("Row count after cleaning",                len(df)),
    ("Duplicate rows remaining",                int(df.duplicated().sum())),
    ("Missing values originally",               int(raw.isnull().sum().sum())),
    ("Missing values remaining",                int(df.isnull().sum().sum())),
    ("Columns",                                 df.shape[1]),
    ("Negative numeric values remaining",       int((df.select_dtypes(include="number") < 0).sum().sum())),
    ("Outlier values flagged by IQR rule (kept)", int(outlier_counts.drop("quality").sum())),
], columns=["Metric", "Value"])

display(quality_report)

df.to_csv("../data/wine_quality_cleaned.csv", index=False)
quality_report.to_csv("../outputs/data_quality_report.csv", index=False)
print("Saved: ../data/wine_quality_cleaned.csv and ../outputs/data_quality_report.csv")

In [ ]:
df = pd.read_csv("../data/wine_quality_cleaned.csv")
print(df.shape)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.histplot(df["alcohol"], kde=True, ax=axes[0, 0], color="skyblue")
axes[0, 0].set_title("Distribution of Alcohol")
axes[0, 0].set_xlabel("Alcohol (% by volume)")

sns.histplot(df["density"], kde=True, ax=axes[0, 1], color="olive")
axes[0, 1].set_title("Distribution of Density")
axes[0, 1].set_xlabel("Density (g/cm3)")

sns.histplot(df["pH"], kde=True, ax=axes[1, 0], color="teal")
axes[1, 0].set_title("Distribution of pH")
axes[1, 0].set_xlabel("pH")

sns.countplot(x="quality", data=df, ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Distribution of Quality")
axes[1, 1].set_xlabel("Quality score")
axes[1, 1].set_ylabel("Number of wines")

plt.tight_layout()
plt.savefig("../outputs/charts/distributions_alcohol_density_ph_quality.png", dpi=150, bbox_inches="tight")
plt.show()

print((df["quality"].value_counts(normalize=True).sort_index() * 100).round(1).rename("% of wines"))

In [ ]:
props = ["alcohol", "density", "pH", "volatile acidity", "citric acid"]

fig, axes = plt.subplots(1, 5, figsize=(16, 4.5))
for ax, col in zip(axes, props):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")
    ax.set_title(col)
    ax.set_ylabel("Value")
plt.suptitle("Boxplots of Chemical Properties (each on its own scale)", y=1.03)
plt.tight_layout()
plt.savefig("../outputs/charts/boxplots_5_properties.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Wine Variables")
plt.savefig("../outputs/charts/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="quality", y="alcohol", color="lightgreen")
plt.title("Alcohol vs Wine Quality")
plt.xlabel("Quality score")
plt.ylabel("Alcohol (% by volume)")
plt.savefig("../outputs/charts/alcohol_vs_quality.png", dpi=150, bbox_inches="tight")
plt.show()

print("Median alcohol by quality score:")
print(df.groupby("quality")["alcohol"].median().round(2))

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="quality", y="volatile acidity", color="salmon")
plt.title("Volatile Acidity vs Wine Quality")
plt.xlabel("Quality score")
plt.ylabel("Volatile acidity (g/dm3)")
plt.savefig("../outputs/charts/volatile_acidity_vs_quality.png", dpi=150, bbox_inches="tight")
plt.show()

print("Median volatile acidity by quality score:")
print(df.groupby("quality")["volatile acidity"].median().round(3))

In [ ]:
cols = ["alcohol", "density", "pH", "volatile acidity", "citric acid"]
z = (df[cols] - df[cols].mean()) / df[cols].std()
z["quality"] = df["quality"]

z.groupby("quality")[cols].mean().plot(kind="bar", figsize=(11, 5), width=0.8)
plt.axhline(0, color="black", linewidth=1)
plt.title("Average Standardised Chemical Properties by Quality Score")
plt.xlabel("Quality score")
plt.ylabel("Mean z-score (0 = dataset average)")
plt.xticks(rotation=0)
plt.legend(title="Property", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../outputs/charts/avg_properties_by_quality.png", dpi=150, bbox_inches="tight")
plt.show()

# Good vs Not Good summary table (raw units)
df["group"] = np.where(df["quality"] >= 7, "Good (>=7)", "Not Good (<7)")
display(df.groupby("group")[cols].mean().round(3))
df = df.drop(columns="group")

In [ ]:
quality_corr = corr["quality"].drop("quality")
quality_corr = quality_corr.reindex(quality_corr.abs().sort_values(ascending=False).index)
display(quality_corr.round(3).to_frame("Correlation with quality"))

plt.figure(figsize=(9, 6))
sns.barplot(x=quality_corr.values, y=quality_corr.index, hue=quality_corr.index,
            palette="Spectral", legend=False)
plt.title("Correlation of Each Variable with Wine Quality")
plt.xlabel("Pearson correlation with quality")
plt.ylabel("Variable")
plt.axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.savefig("../outputs/charts/quality_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

 Observations (supported by the charts above)

1. Quality is concentrated in the middle scores.About 92% of wines score 5, 6 or 7 (29.7%, 45.1% and 17.4%). Scores 3 and 9 are very rare (0.5% and 0.1%), so extreme-quality wines are poorly represented.
2. Alcohol is the variable most strongly associated with quality (r = +0.46). Among the common scores the median alcohol climbs steadily from 9.6% (score 5) to 10.5% (score 6), 11.5% (score 7), 12.2% (score 8) and 12.5% (score 9). Wines rated 7 or above average 11.58% alcohol compared with 10.33% for the rest. (The few score-3 and score-4 wines do not follow the pattern, but there are only 20 and 153 of them.)
3. Density is negatively associated with quality(r = -0.34), but density is itself driven by alcohol (r = -0.76 with alcohol) and by sugar (r = +0.82 with residual sugar). Its link with quality therefore overlaps with those variables, a warning about multicollinearityfor  the regression models.
4. Volatile acidity shows a weak negative association (r = -0.19): the median falls from 0.32 g/dm3 at score 4 to 0.25 at scores 6 and 7, which fits the idea that a vinegar-like taste lowers quality. The effect is modest: the boxes for neighbouring scores overlap a lot and the median does not keep falling for the highest scores.
5. Chlorides (r = -0.22) and total sulfur dioxide (r = -0.18) are the next strongest negative associations; higher salt and sulfur levels tend to accompany lower scores.
6.Citric acid (r = 0.01), free sulfur dioxide (r = 0.01) and sulphates (r = 0.05) have almost no linear association with quality on their own.
7. The boxplots show outliers in several variables (e.g. citric acid, chlorides, residual sugar). They are plausible measurements and were kept (see Part A).
8. No single variable explains quality well: even alcohol explains only about 21% of the variance (r squared = 0.21). We should expect only moderate predictive power from the models.

Conclusion: alcohol, density, chlorides, volatile acidity and total sulfur dioxide show the strongest observed associations with wine quality. Correlation does not prove cause and effect.